In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     PPO  ·  LunarLander-v3 (continuous)  ·  TRAINING           ║
# ╚══════════════════════════════════════════════════════════════════╝
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import sys

# ── Config ────────────────────────────────────────────────────────────────────
STATE_DIM  = 8
ACTION_DIM = 2
MAX_ACTION = 1.0

learning_rate   = 3e-4
gamma           = 0.99
lmbda           = 0.9
eps_clip        = 0.2
K_epoch         = 10
rollout_len     = 3
buffer_size     = 10
minibatch_size  = 32

TOTAL_EPISODES  = 4000
EVAL_INTERVAL   = 50
SAVE_INTERVAL   = 500
N_EVAL_EPISODES = 5

FINAL_PATH = 'ppo_lunar_lander_final.pt'
BEST_PATH  = 'ppo_lunar_lander_best.pt'
LOG_DIR    = f'runs/PPO_lunar_lander_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}', flush=True)
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}', flush=True)
print(f'Log dir: {LOG_DIR}', flush=True)

# ── PPO Model ─────────────────────────────────────────────────────────────────
class PPO(nn.Module):
    def __init__(self):
        super().__init__()
        self.data = []

        self.fc1    = nn.Linear(STATE_DIM, 256)
        self.fc2    = nn.Linear(256, 256)
        self.fc_mu  = nn.Linear(256, ACTION_DIM)
        self.fc_std = nn.Linear(256, ACTION_DIM)
        self.fc_v   = nn.Linear(256, 1)

        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        self.optimization_step = 0

    def pi(self, x, softmax_dim=0):
        x   = F.relu(self.fc1(x))
        x   = F.relu(self.fc2(x))
        mu  = MAX_ACTION * torch.tanh(self.fc_mu(x))
        std = F.softplus(self.fc_std(x)) + 1e-5
        return mu, std

    def v(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc_v(x)

    def put_data(self, rollout):
        self.data.append(rollout)

    def make_batch(self):
        s_buf, a_buf, r_buf, sp_buf, d_buf, lp_buf = [], [], [], [], [], []
        for rollout in self.data:
            for s, a, r, sp, done_mask, lp in rollout:
                s_buf.append(s)
                a_buf.append(a)
                r_buf.append([r])
                sp_buf.append(sp)
                d_buf.append([done_mask])
                lp_buf.append(lp)          # shape (1,)
        self.data = []

        N, T = minibatch_size * buffer_size, rollout_len
        s  = torch.FloatTensor(np.array(s_buf)).to(device).view(N, T, STATE_DIM)
        a  = torch.FloatTensor(np.array(a_buf)).to(device).view(N, T, ACTION_DIM)
        r  = torch.FloatTensor(r_buf).to(device).view(N, T, 1)
        sp = torch.FloatTensor(np.array(sp_buf)).to(device).view(N, T, STATE_DIM)
        d  = torch.FloatTensor(d_buf).to(device).view(N, T, 1)
        lp = torch.FloatTensor(np.array(lp_buf)).to(device).view(N, T, 1)

        idx = torch.randperm(N)
        mini_batches = []
        for i in range(buffer_size):
            mb = idx[i * minibatch_size : (i+1) * minibatch_size]
            mini_batches.append((s[mb], a[mb], r[mb], sp[mb], d[mb], lp[mb]))
        return mini_batches

    def calc_advantage(self, data):
        data_with_adv = []
        for s, a, r, sp, done_mask, old_lp in data:
            with torch.no_grad():
                td_target = r + gamma * self.v(sp) * done_mask
                delta     = td_target - self.v(s)          # [B, T, 1]

                advantage = torch.zeros_like(delta)
                adv = torch.zeros(minibatch_size, 1, 1, device=device)
                for t in range(rollout_len - 1, -1, -1):
                    adv = delta[:, t:t+1, :] + gamma * lmbda * done_mask[:, t:t+1, :] * adv
                    advantage[:, t:t+1, :] = adv

            data_with_adv.append((s, a, r, sp, done_mask, old_lp, td_target, advantage))
        return data_with_adv

    def train_net(self):
        if len(self.data) != minibatch_size * buffer_size:
            return

        data = self.make_batch()
        data = self.calc_advantage(data)

        for _ in range(K_epoch):
            for mini_batch in data:
                s, a, r, sp, done_mask, old_log_prob, td_target, advantage = mini_batch

                mu, std  = self.pi(s, softmax_dim=2)
                dist     = Normal(mu, std)
                log_prob = dist.log_prob(a).sum(-1, keepdim=True)   # [B, T, 1]

                ratio = torch.exp(log_prob - old_log_prob)
                surr1 = ratio * advantage
                surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantage

                loss = (-torch.min(surr1, surr2)
                        + F.smooth_l1_loss(self.v(s), td_target, reduction='none'))

                self.optimizer.zero_grad()
                loss.mean().backward()
                nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                self.optimizer.step()
                self.optimization_step += 1

# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(model):
    eval_env = gym.make('LunarLander-v3', continuous=True)
    rewards  = []
    for _ in range(N_EVAL_EPISODES):
        s, _  = eval_env.reset()
        done  = False
        ep_r  = 0.0
        while not done:
            with torch.no_grad():
                s_t   = torch.FloatTensor(s).to(device)
                mu, _ = model.pi(s_t)
                a     = mu.clamp(-MAX_ACTION, MAX_ACTION).cpu().numpy()
            s, r, done, truncated, _ = eval_env.step(a)
            done = done or truncated
            ep_r += r
        rewards.append(ep_r)
    eval_env.close()
    return float(np.mean(rewards))

# ── Main Training Loop ────────────────────────────────────────────────────────
def main():
    env    = gym.make('LunarLander-v3', continuous=True)
    model  = PPO().to(device)
    writer = SummaryWriter(log_dir=LOG_DIR)

    s, _      = env.reset()
    ep_reward = 0.0
    ep_num    = 0
    best_eval = float('-inf')

    print(f'\n{"="*60}', flush=True)
    print(f'  PPO  |  LunarLander-v3 continuous', flush=True)
    print(f'  Total episodes : {TOTAL_EPISODES:,}', flush=True)
    print(f'  rollout_len={rollout_len}  buffer_size={buffer_size}  minibatch={minibatch_size}', flush=True)
    print(f'  Train every {rollout_len * minibatch_size * buffer_size} steps '
          f'({minibatch_size * buffer_size} rollouts)', flush=True)
    print(f'  Eval interval  : every {EVAL_INTERVAL} ep  ({N_EVAL_EPISODES} ep eval)', flush=True)
    print(f'{"="*60}\n', flush=True)

    while ep_num < TOTAL_EPISODES:
        rollout = []
        for t in range(rollout_len):
            with torch.no_grad():
                s_t    = torch.FloatTensor(s).to(device)
                mu, std = model.pi(s_t)
                dist   = Normal(mu, std)
                a_t    = dist.sample().clamp(-MAX_ACTION, MAX_ACTION)
                lp     = dist.log_prob(a_t).sum(-1, keepdim=True).cpu().numpy()  # (1,)

            a_np = a_t.cpu().numpy()
            s_, r, done, truncated, _ = env.step(a_np)
            terminal = done or truncated

            rollout.append((s.copy(), a_np.copy(), float(r), s_.copy(),
                            0.0 if terminal else 1.0, lp.copy()))
            ep_reward += r
            s = s_

            if terminal:
                writer.add_scalar('Reward/episode', ep_reward, ep_num)
                if ep_num % 10 == 0:
                    print(f'  epi {ep_num:5d} | reward {ep_reward:8.1f} | '
                          f'optim_step {model.optimization_step:,}', flush=True)
                ep_num    += 1
                ep_reward  = 0.0

                if ep_num >= TOTAL_EPISODES:
                    break

                if ep_num % EVAL_INTERVAL == 0:
                    avg_r = evaluate(model)
                    writer.add_scalar('Reward/eval', avg_r, ep_num)
                    print(f'  [eval] epi {ep_num:5d} | avg {N_EVAL_EPISODES} ep: {avg_r:8.1f}', flush=True)
                    if avg_r > best_eval:
                        best_eval = avg_r
                        torch.save(model.state_dict(), BEST_PATH)
                        print(f'  [best] {best_eval:.1f} -> {BEST_PATH}', flush=True)
                    writer.add_scalar('Reward/best', best_eval, ep_num)

                if ep_num % SAVE_INTERVAL == 0 and ep_num > 0:
                    torch.save(model.state_dict(), FINAL_PATH)
                    print(f'  [saved] epi {ep_num:,} -> {FINAL_PATH}', flush=True)

                s, _ = env.reset()

        if len(rollout) == rollout_len:
            model.put_data(rollout)
            model.train_net()

    env.close()
    writer.close()
    torch.save(model.state_dict(), FINAL_PATH)
    print(f'\nTraining done.', flush=True)
    print(f'  Final -> {FINAL_PATH}', flush=True)
    print(f'  Best  -> {BEST_PATH}  (eval reward: {best_eval:.1f})', flush=True)
    print(f'  TensorBoard: tensorboard --logdir {LOG_DIR}', flush=True)

main()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     PPO  ·  LunarLander-v3 (continuous)  ·  INFERENCE          ║
# ╚══════════════════════════════════════════════════════════════════╝
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ── Config (must match training) ──────────────────────────────────────────────
STATE_DIM  = 8
ACTION_DIM = 2
MAX_ACTION = 1.0

FINAL_PATH = 'ppo_lunar_lander_final.pt'
BEST_PATH  = 'ppo_lunar_lander_best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── PPO Model (same architecture, no optimizer) ───────────────────────────────
class PPO(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1    = nn.Linear(STATE_DIM, 256)
        self.fc2    = nn.Linear(256, 256)
        self.fc_mu  = nn.Linear(256, ACTION_DIM)
        self.fc_std = nn.Linear(256, ACTION_DIM)
        self.fc_v   = nn.Linear(256, 1)

    def pi(self, x):
        x   = F.relu(self.fc1(x))
        x   = F.relu(self.fc2(x))
        mu  = MAX_ACTION * torch.tanh(self.fc_mu(x))
        std = F.softplus(self.fc_std(x)) + 1e-5
        return mu, std

    def v(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc_v(x)

# ── Inference ─────────────────────────────────────────────────────────────────
def run_inference(save_path=FINAL_PATH, n_episodes=5):
    env = gym.make('LunarLander-v3', continuous=True, render_mode='human')

    model = PPO().to(device)
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()

    model_tag = 'final' if save_path == FINAL_PATH else 'best'
    print(f'[{model_tag}] loaded from {save_path}')
    print(f'Running {n_episodes} episodes (deterministic mu)...\n')

    var_labels    = ['x pos', 'y pos', 'x vel', 'y vel',
                     'angle', 'ang vel', 'leg L', 'leg R']
    action_labels = ['main engine', 'left/right engine']

    for ep in range(n_episodes):
        s, _         = env.reset()
        done         = False
        obs_history  = []
        act_history  = []
        val_history  = []
        total_reward = 0.0

        with torch.no_grad():
            while not done:
                obs_history.append(s.copy())
                s_t   = torch.FloatTensor(s).to(device)
                mu, _ = model.pi(s_t)
                a     = mu.clamp(-MAX_ACTION, MAX_ACTION).cpu().numpy()
                val_history.append(model.v(s_t).cpu().item())
                act_history.append(a.copy())
                s, r, done, truncated, _ = env.step(a)
                done = done or truncated
                total_reward += r

        steps = len(obs_history)
        print(f'Episode {ep+1:2d}/{n_episodes} | reward: {total_reward:8.1f} | steps: {steps}')

        obs = np.array(obs_history)   # (T, 8)
        act = np.array(act_history)   # (T, 2)
        val = np.array(val_history)   # (T,)
        t   = np.arange(steps)

        # ── Observation plot (4×2) ─────────────────────────────────────────────
        fig, axes = plt.subplots(4, 2, figsize=(14, 12))
        for ax, col, label in zip(axes.flatten(), range(8), var_labels):
            ax.plot(t, obs[:, col], linewidth=1.5)
            ax.set_title(label, fontsize=11)
            ax.set_xlabel('Time step')
            ax.grid(True, alpha=0.3)
        plt.suptitle(
            f'PPO [{model_tag}] — Ep {ep+1}  Observations  (reward={total_reward:.1f})',
            fontsize=13, y=1.01)
        plt.tight_layout()
        obs_path = f'ppo_{model_tag}_obs_ep{ep+1}.png'
        plt.savefig(obs_path, dpi=150, bbox_inches='tight')
        plt.show()

        # ── Action plot (2×1) ─────────────────────────────────────────────────
        fig, axes = plt.subplots(2, 1, figsize=(12, 5))
        for ax, col, label in zip(axes, range(2), action_labels):
            ax.plot(t, act[:, col], linewidth=1.5, color='mediumseagreen')
            ax.set_title(label, fontsize=11)
            ax.set_xlabel('Time step')
            ax.set_ylim(-1.05, 1.05)
            ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
            ax.grid(True, alpha=0.3)
        plt.suptitle(
            f'PPO [{model_tag}] — Ep {ep+1}  Actions  (reward={total_reward:.1f})',
            fontsize=13, y=1.01)
        plt.tight_layout()
        act_path = f'ppo_{model_tag}_act_ep{ep+1}.png'
        plt.savefig(act_path, dpi=150, bbox_inches='tight')
        plt.show()

        # ── Value estimate plot ────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(t, val, linewidth=1.5, color='darkorange')
        ax.set_title('Critic Value Estimate  V(s)', fontsize=11)
        ax.set_xlabel('Time step')
        ax.grid(True, alpha=0.3)
        plt.suptitle(
            f'PPO [{model_tag}] — Ep {ep+1}  Value  (reward={total_reward:.1f})',
            fontsize=13, y=1.05)
        plt.tight_layout()
        val_path = f'ppo_{model_tag}_val_ep{ep+1}.png'
        plt.savefig(val_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f'  obs -> {obs_path}  |  act -> {act_path}  |  val -> {val_path}\n')

    env.close()
    print('Done.')

# ── 실행 ──────────────────────────────────────────────────────────────────────
run_inference(save_path=FINAL_PATH)       # 최종 모델, 5 에피소드
# run_inference(save_path=BEST_PATH)      # best 모델로 추론하려면 이 줄 실행